In [ ]:
code = r'''
import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet

class Planet:
    def __init__(self, id, owner, x, y, radius, ships, production):
        self.id=id; self.owner=owner; self.x=x; self.y=y
        self.radius=radius; self.ships=ships; self.production=production

def dist(a,b):
    return math.hypot(a.x-b.x, a.y-b.y)

def travel_time(a,b,ships):
    ships = max(1, ships)
    speed = 1.0 + (6.0 - 1.0) * ((math.log(ships) / math.log(1000)) ** 1.5)
    return dist(a,b) / max(speed, 1e-6)

def score(src,tgt):
    return tgt.production*5 - dist(src,tgt)*0.8 - tgt.ships*1.2

def future_ships(tgt, t):
    return tgt.ships + tgt.production * t

def agent(obs):
    try:
        moves = []
        player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
        raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets

        # Parse into named tuples for readable field access:
        #   Planet(id, owner, x, y, radius, ships, production)
        #   owner == -1 means neutral, 0-3 are player IDs
        planets = [Planet(*p) for p in raw_planets]
        my_planets = [p for p in planets if p.owner == player]
        targets = [p for p in planets if p.owner != player]

        if not targets:
            return moves

        for mine in my_planets:
            # Find the nearest planet we don't own
            nearest = None
            min_dist = float("inf")
            for t in targets:
                dist = math.sqrt((mine.x - t.x) ** 2 + (mine.y - t.y) ** 2)
                if dist < min_dist:
                    min_dist = dist
                    nearest = t

            if nearest is None:
                continue

            # We need to send more ships than the target has to capture it.
            # Exactly target_ships + 1 guarantees the takeover.
            ships_needed = nearest.ships + 1

            # Only launch if we can afford it — otherwise keep accumulating
            if mine.ships >= ships_needed:
                # atan2(dy, dx) gives the angle from our planet to the target
                angle = math.atan2(nearest.y - mine.y, nearest.x - mine.x)
                moves.append([mine.id, angle, ships_needed])

        return moves
    
    except:
        planets = [Planet(*p) for p in obs.get("planets", [])]
        player = obs.get("player", 0)

        my_planets = [p for p in planets if p.owner == player and p.ships > 1]
        targets = [p for p in planets if p.owner != player]

        moves = []

        if not my_planets or not targets:
            return moves

        for src in my_planets:
            best = None
            best_score = -1e9

            for tgt in targets:
                if tgt.id == src.id:
                    continue
                s = score(src, tgt)
                if s > best_score:
                    best_score = s
                    best = tgt

            if best is None:
                continue

            t = travel_time(src, best, src.ships)
            needed = future_ships(best, t) + 1

            if src.ships <= needed:
                continue

            send = int(min(src.ships - 1, needed + 5))
            if send <= 0:
                continue

            angle = math.atan2(best.y - src.y, best.x - src.x)

            if not math.isfinite(angle):
                continue

            moves.append([int(src.id), float(angle), int(send)])

        return moves
'''

with open("/kaggle/working/submission.py","w") as f:
    f.write(code)

print("submission.py created")